In [1]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.config import *
from src.utils import setup_logger
import faiss
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer
import torch
from typing import List, Dict
import re

logger = setup_logger()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Setup complete")
print(f"Device: {device}")

Setup complete
Device: cuda


In [2]:
print("=== Loading Chunks and Embeddings ===\n")

chunks_file = PROCESSED_DATA_DIR / "chunks.pkl"
embeddings_file = PROCESSED_DATA_DIR / "embeddings.npy"

with open(chunks_file, 'rb') as f:
    all_chunks = pickle.load(f)

all_embeddings = np.load(embeddings_file)

print(f"✓ Loaded {len(all_chunks)} chunks")
print(f"✓ Loaded {len(all_embeddings)} embeddings")
print(f"✓ Embedding dimension: {all_embeddings.shape[1]}")

=== Loading Chunks and Embeddings ===

✓ Loaded 2908 chunks
✓ Loaded 2908 embeddings
✓ Embedding dimension: 384


In [3]:
print("=== Setting Up Crisis Detection ===\n")

CRISIS_KEYWORDS = [
    # Suicide-related
    "suicide", "suicidal", "kill myself", "end my life", "want to die",
    "don't want to live", "no reason to live", "better off dead",
    "take my own life", "end it all", "killing myself",
    
    # Self-harm
    "cut myself", "hurt myself", "self harm", "self-harm", "cutting",
    "burn myself", "harm myself",
    
    # Immediate danger
    "going to die", "plan to die", "goodbye forever", "final goodbye",
    "overdose", "pills", "jump off", "hang myself",
    
    # Desperation
    "can't go on", "no way out", "hopeless", "give up on life",
    "nothing left", "everyone would be better", "burden to everyone"
]

INDIAN_CRISIS_RESOURCES = """
🚨 IMMEDIATE CRISIS SUPPORT - INDIA 🚨

Please reach out to professional crisis support immediately:

📞 **24/7 Helplines:**
- Vandrevala Foundation: 1860-2662-345 / 1800-2333-330
- AASRA: 91-22-27546669
- iCall (Tata Institute): 022-25521111 / 9152987821
- Snehi: 91-22-27546669
- NIMHANS: 080-46110007

💬 **Text Support:**
- iCall WhatsApp: 9152987821

🌐 **Online:**
- www.mpowerminds.com
- www.vandrevalafoundation.com

You are not alone. These trained professionals are here to help you right now.
"""

print(f"✓ {len(CRISIS_KEYWORDS)} crisis keywords configured")
print("✓ Indian crisis resources loaded")

=== Setting Up Crisis Detection ===

✓ 33 crisis keywords configured
✓ Indian crisis resources loaded


In [4]:
def detect_crisis(text: str, user_context: str = "general") -> Dict:
    """
    Detect crisis indicators tailored to different demographics.
    
    Args:
        text: User's message
        user_context: 'student', 'athlete', 'professional', or 'general'
    
    Returns:
        dict with crisis info and tailored resources
    """
    text_lower = text.lower()
    matched = []
    context_indicators = []
    
    # Universal crisis keywords
    for keyword in CRISIS_KEYWORDS:
        if keyword in text_lower:
            matched.append(keyword)
    
    # Student-specific crisis indicators
    student_crisis = [
        "academic pressure", "exam stress", "failed exam", "can't handle studies",
        "disappointing parents", "college stress", "academic failure",
        "too much pressure", "parents will kill me", "can't face results"
    ]
    
    # Athlete-specific crisis indicators
    athlete_crisis = [
        "career ending injury", "lost my sport", "can't compete anymore",
        "failed selection", "dropped from team", "performance anxiety",
        "let my team down", "coach disappointed", "injury ruined everything"
    ]
    
    # Professional-specific crisis indicators
    professional_crisis = [
        "lost my job", "fired", "career is over", "work pressure unbearable",
        "can't meet targets", "failing at work", "burnout", "toxic workplace",
        "financial crisis", "drowning in debt", "can't pay bills"
    ]
    
    # Check context-specific indicators
    if user_context == "student":
        for keyword in student_crisis:
            if keyword in text_lower:
                context_indicators.append(keyword)
    elif user_context == "athlete":
        for keyword in athlete_crisis:
            if keyword in text_lower:
                context_indicators.append(keyword)
    elif user_context == "professional":
        for keyword in professional_crisis:
            if keyword in text_lower:
                context_indicators.append(keyword)
    
    # Determine crisis level
    total_matches = len(matched) + len(context_indicators)
    is_crisis = total_matches > 0
    
    if total_matches >= 3:
        confidence = "high"
    elif total_matches >= 2:
        confidence = "medium"
    elif total_matches >= 1:
        confidence = "low"
    else:
        confidence = "none"
    
    # Tailored resources based on context
    if is_crisis:
        if user_context == "student":
            resource_msg = INDIAN_CRISIS_RESOURCES + """
📚 **Student-Specific Support:**
- Your college counseling center
- Academic stress helplines
- Student mental health services
- Remember: One exam/grade does not define your worth
"""
        elif user_context == "athlete":
            resource_msg = INDIAN_CRISIS_RESOURCES + """
🏃 **Athlete-Specific Support:**
- Sports psychologists
- Career transition counseling
- Athlete mental health programs
- Remember: You are more than your sport
"""
        elif user_context == "professional":
            resource_msg = INDIAN_CRISIS_RESOURCES + """
💼 **Professional Support:**
- Employee Assistance Programs (EAP)
- Workplace counseling services
- Financial counseling helplines
- Career transition support
- Remember: Your job does not define your value
"""
        else:
            resource_msg = INDIAN_CRISIS_RESOURCES
    else:
        resource_msg = None
    
    return {
        "is_crisis": is_crisis,
        "matched_keywords": matched,
        "context_indicators": context_indicators,
        "confidence": confidence,
        "user_context": user_context,
        "resource_message": resource_msg
    }

print("✓ Advanced crisis detection function defined")

# Test with different contexts
test_cases = [
    ("I'm feeling a bit down today", "general"),
    ("I want to end my life", "general"),
    ("Failed my exams and can't face my parents", "student"),
    ("Career ending injury, I've lost everything", "athlete"),
    ("Lost my job, drowning in debt, want to die", "professional"),
]

print("\nTesting crisis detection across demographics:\n")
for test_text, context in test_cases:
    result = detect_crisis(test_text, context)
    print(f"Text: '{test_text}'")
    print(f"Context: {context}")
    print(f"  Crisis: {result['is_crisis']} | Confidence: {result['confidence']}")
    print(f"  Universal matches: {result['matched_keywords']}")
    print(f"  Context matches: {result['context_indicators']}")
    print()

✓ Advanced crisis detection function defined

Testing crisis detection across demographics:

Text: 'I'm feeling a bit down today'
Context: general
  Crisis: False | Confidence: none
  Universal matches: []
  Context matches: []

Text: 'I want to end my life'
Context: general
  Crisis: True | Confidence: low
  Universal matches: ['end my life']
  Context matches: []

Text: 'Failed my exams and can't face my parents'
Context: student
  Crisis: False | Confidence: none
  Universal matches: []
  Context matches: []

Text: 'Career ending injury, I've lost everything'
Context: athlete
  Crisis: True | Confidence: low
  Universal matches: []
  Context matches: ['career ending injury']

Text: 'Lost my job, drowning in debt, want to die'
Context: professional
  Crisis: True | Confidence: high
  Universal matches: ['want to die']
  Context matches: ['lost my job', 'drowning in debt']



In [5]:
print("=== Building FAISS Vector Index ===\n")

embedding_dim = all_embeddings.shape[1]

index = faiss.IndexFlatL2(embedding_dim)

index.add(all_embeddings.astype('float32'))

print(f"✓ FAISS index created")
print(f"✓ Index type: Flat L2 (exact search)")
print(f"✓ Indexed {index.ntotal} vectors")
print(f"✓ Dimension: {embedding_dim}")

=== Building FAISS Vector Index ===

✓ FAISS index created
✓ Index type: Flat L2 (exact search)
✓ Indexed 2908 vectors
✓ Dimension: 384


In [6]:
print("=== Saving FAISS Index ===\n")

index_file = VECTOR_STORE_DIR / "faiss_index.bin"
faiss.write_index(index, str(index_file))

print(f"✓ FAISS index saved to: {index_file}")
print(f"✓ File size: {index_file.stat().st_size / 1024 / 1024:.2f} MB")

=== Saving FAISS Index ===

✓ FAISS index saved to: C:\Users\patel\Documents\chitraksha\vector_store\faiss_index\faiss_index.bin
✓ File size: 4.26 MB


In [7]:
print("=== Loading Embedding Model ===\n")

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)

print(f"✓ Model loaded on {device}")
print(f"✓ Ready for semantic search")

=== Loading Embedding Model ===

✓ Model loaded on cuda
✓ Ready for semantic search


In [8]:
def semantic_search(query: str, top_k: int = 5, user_context: str = "general") -> Dict:
    """
    Perform semantic search with crisis detection.
    
    Args:
        query: User's question/message
        top_k: Number of results to return
        user_context: User demographic context
        
    Returns:
        dict with results and crisis info
    """
    # 1. Crisis detection first
    crisis_result = detect_crisis(query, user_context)
    
    # 2. Convert query to embedding
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)[0]
    query_embedding = query_embedding.astype('float32').reshape(1, -1)
    
    # 3. Search FAISS index
    distances, indices = index.search(query_embedding, top_k)
    
    # 4. Retrieve matching chunks
    results = []
    for i, idx in enumerate(indices[0]):
        chunk = all_chunks[idx]
        results.append({
            'rank': i + 1,
            'chunk': chunk,
            'distance': float(distances[0][i]),
            'similarity': 1 / (1 + distances[0][i])  # Convert distance to similarity score
        })
    
    return {
        'query': query,
        'crisis_detected': crisis_result['is_crisis'],
        'crisis_info': crisis_result if crisis_result['is_crisis'] else None,
        'results': results,
        'user_context': user_context
    }

print("✓ Semantic search function defined")

✓ Semantic search function defined


In [9]:
print("=== Testing Semantic Search ===\n")

test_queries = [
    ("I'm feeling anxious about my upcoming presentation", "professional"),
    ("Can't sleep because of exam stress", "student"),
    ("I want to end my life", "general"),
]

for query, context in test_queries:
    print(f"Query: '{query}'")
    print(f"Context: {context}")
    
    search_results = semantic_search(query, top_k=3, user_context=context)
    
    if search_results['crisis_detected']:
        print("  🚨 CRISIS DETECTED")
        print(f"  Confidence: {search_results['crisis_info']['confidence']}")
        print(f"  Keywords: {search_results['crisis_info']['matched_keywords']}")
    else:
        print("  ✓ No crisis detected")
    
    print("\n  Top 3 Results:")
    for result in search_results['results']:
        print(f"    {result['rank']}. Similarity: {result['similarity']:.3f}")
        print(f"       Source: {result['chunk']['source']}")
        print(f"       Text: {result['chunk']['text'][:100]}...")
    
    print("\n" + "="*60 + "\n")

=== Testing Semantic Search ===

Query: 'I'm feeling anxious about my upcoming presentation'
Context: professional
  ✓ No crisis detected

  Top 3 Results:
    1. Similarity: 0.488
       Source: kaggle_mental_health
       Text: User: Probably because my exams are approaching. I feel stressed out because I don't think I've prep...
    2. Similarity: 0.488
       Source: kaggle_mental_health
       Text: User: Probably because my exams are approaching. I feel stressed out because I don't think I've prep...
    3. Similarity: 0.482
       Source: kaggle_mental_health
       Text: User: I feel so anxious. Counselor: Can you tell me more about this feeling?...


Query: 'Can't sleep because of exam stress'
Context: student
  🚨 CRISIS DETECTED
  Confidence: low
  Keywords: []

  Top 3 Results:
    1. Similarity: 0.546
       Source: kaggle_mental_health
       Text: User: I guess not. All I can think about are my exams. Counselor: That's no problem. I can see why y...
    2. Similarity: 0.5

In [10]:
def advanced_search(query: str, top_k: int = 5, user_context: str = "general", 
                   filter_source: str = None, min_similarity: float = 0.3) -> Dict:
    """
    Advanced semantic search with filtering options.
    
    Args:
        query: User's question
        top_k: Number of results
        user_context: User demographic
        filter_source: Only return chunks from this source (optional)
        min_similarity: Minimum similarity threshold
    """
    crisis_result = detect_crisis(query, user_context)
    
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)[0]
    query_embedding = query_embedding.astype('float32').reshape(1, -1)
    
    # Search more results to allow for filtering
    search_k = top_k * 3 if filter_source else top_k
    distances, indices = index.search(query_embedding, search_k)
    
    results = []
    for i, idx in enumerate(indices[0]):
        chunk = all_chunks[idx]
        similarity = 1 / (1 + distances[0][i])
        
        # Apply filters
        if filter_source and chunk['source'] != filter_source:
            continue
        if similarity < min_similarity:
            continue
        
        results.append({
            'rank': len(results) + 1,
            'chunk': chunk,
            'distance': float(distances[0][i]),
            'similarity': similarity
        })
        
        if len(results) >= top_k:
            break
    
    return {
        'query': query,
        'crisis_detected': crisis_result['is_crisis'],
        'crisis_info': crisis_result if crisis_result['is_crisis'] else None,
        'results': results,
        'user_context': user_context,
        'filters_applied': {
            'source': filter_source,
            'min_similarity': min_similarity
        }
    }

print("✓ Advanced search function defined")

✓ Advanced search function defined


In [11]:
import json
from datetime import datetime

metadata = {
    "created_timestamp": datetime.now().isoformat(),
    "total_chunks": len(all_chunks),
    "total_embeddings": len(all_embeddings),
    "embedding_dimension": int(all_embeddings.shape[1]),
    "embedding_model": EMBEDDING_MODEL_NAME,
    "index_type": "FAISS IndexFlatL2",
    "crisis_keywords_count": len(CRISIS_KEYWORDS),
    "crisis_detection_enabled": True,
    "supported_contexts": ["general", "student", "athlete", "professional"],
}

metadata_file = VECTOR_STORE_DIR / "vector_store_metadata.json"
with open(metadata_file, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

print("=== Vector Store Metadata Saved ===")
print(f"Location: {metadata_file}")
print("\nMetadata:")
print(json.dumps(metadata, indent=2))

=== Vector Store Metadata Saved ===
Location: C:\Users\patel\Documents\chitraksha\vector_store\faiss_index\vector_store_metadata.json

Metadata:
{
  "created_timestamp": "2025-12-07T14:42:28.356576",
  "total_chunks": 2908,
  "total_embeddings": 2908,
  "embedding_dimension": 384,
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "index_type": "FAISS IndexFlatL2",
  "crisis_keywords_count": 33,
  "crisis_detection_enabled": true,
  "supported_contexts": [
    "general",
    "student",
    "athlete",
    "professional"
  ]
}


In [12]:
print("=== Testing Complete RAG Retrieval ===\n")

def format_rag_context(search_results: Dict) -> str:
    """Format search results into context for LLM."""
    
    if search_results['crisis_detected']:
        context = "🚨 CRISIS SITUATION DETECTED 🚨\n\n"
        context += search_results['crisis_info']['resource_message']
        context += "\n\n--- Relevant Support Information ---\n\n"
    else:
        context = "--- Relevant Information from Knowledge Base ---\n\n"
    
    for result in search_results['results']:
        chunk = result['chunk']
        context += f"[Source: {chunk['source']} | Relevance: {result['similarity']:.2f}]\n"
        context += f"{chunk['text']}\n\n"
    
    return context

# Test scenarios
test_scenarios = [
    ("How do I manage work stress?", "professional"),
    ("Failed my exam and feeling hopeless", "student"),
    ("I want to kill myself", "general"),
]

for query, context in test_scenarios:
    print("="*60)
    print(f"Query: '{query}'")
    print(f"Context: {context}\n")
    
    search_results = semantic_search(query, top_k=3, user_context=context)
    formatted_context = format_rag_context(search_results)
    
    print(formatted_context)
    print("="*60 + "\n")

=== Testing Complete RAG Retrieval ===

Query: 'How do I manage work stress?'
Context: professional

--- Relevant Information from Knowledge Base ---

[Source: hf_chatbot | Relevance: 0.63]
<HUMAN>: How to manage stress? <ASSISTANT>: Here are some personalized suggestions to help you cope with stress: 1. Identify Triggers: Take some time to identify what triggers your stress. This awareness can help you anticipate stressful situations and prepare for them. 2. Practice Deep Breathing: When you feel stressed, try taking slow, deep breaths. Inhale deeply for a count of four, hold for four, and then exhale for four. This simple technique can help calm your nervous system. 3. Stay Active: Engaging

[Source: hf_chatbot | Relevance: 0.63]
<HUMAN>: How to manage stress? <ASSISTANT>: Here are some personalized suggestions to help you cope with stress: 1. Identify Triggers: Take some time to identify what triggers your stress. This awareness can help you anticipate stressful situations and prepa

In [13]:
print("="*60)
print("PHASE 4: VECTOR STORE + CRISIS DETECTION COMPLETE")
print("="*60)

print("\n✓ FAISS index built and saved")
print(f"✓ {len(all_chunks):,} chunks indexed")
print("✓ Semantic search functional")
print("✓ Crisis detection integrated")
print("✓ Multi-demographic support enabled")
print("✓ Indian crisis resources configured")

print("\n" + "="*60)
print("READY FOR PHASE 5: LLM INTEGRATION")
print("="*60)

print("\nWhat we've built:")
print("• Vector store for fast semantic retrieval")
print("• Crisis detection (keyword + context-aware)")
print("• Demographic-specific responses (student/athlete/professional)")
print("• RAG context formatting for LLM")

print("\nNext phase:")
print("1. Save this notebook (Ctrl+S)")
print("2. Create new notebook: 05_llm_integration.ipynb")
print("3. Load local LLM (Phi-3.5-mini)")
print("4. Create empathetic conversation prompts")
print("5. Integrate RAG retrieval with LLM generation")
print("6. Build crisis response protocol")

PHASE 4: VECTOR STORE + CRISIS DETECTION COMPLETE

✓ FAISS index built and saved
✓ 2,908 chunks indexed
✓ Semantic search functional
✓ Crisis detection integrated
✓ Multi-demographic support enabled
✓ Indian crisis resources configured

READY FOR PHASE 5: LLM INTEGRATION

What we've built:
• Vector store for fast semantic retrieval
• Crisis detection (keyword + context-aware)
• Demographic-specific responses (student/athlete/professional)
• RAG context formatting for LLM

Next phase:
1. Save this notebook (Ctrl+S)
2. Create new notebook: 05_llm_integration.ipynb
3. Load local LLM (Phi-3.5-mini)
4. Create empathetic conversation prompts
5. Integrate RAG retrieval with LLM generation
6. Build crisis response protocol
